In [1]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
URL = ('http://archive.ics.uci.edu/static/public/222/bank+marketing.zip')

outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read('bank-additional.zip')))
df = pd.read_csv(
    inner.open('bank-additional/bank-additional.csv'), sep=';'
)
df.to_csv('bank.csv', sep=';', index=False)
print(df.shape) # (4119, 21)

(4119, 21)


In [2]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-null   object 
 5   housing         4119 non-null   object 
 6   loan            4119 non-null   object 
 7   contact         4119 non-null   object 
 8   month           4119 non-null   object 
 9   day_of_week     4119 non-null   object 
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   object 
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

In [7]:
print(df['y'].value_counts())
print((df['y'] == 'yes').mean())

y
no     3668
yes     451
Name: count, dtype: int64
0.10949259529011895


In [9]:
df.groupby('y')['duration'].mean().round(1)

,duration
y,
no,219.4
yes,560.8


In [10]:
df = df.drop(columns=['duration'])

In [11]:
df.shape

(4119, 20)

In [14]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=None)
from sklearn.model_selection import train_test_split
X = df.drop(columns=['y'])
y = df['y']
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dt.fit(X_train, y_train)
print(dt.score(X_train, y_train))
print(dt.score(X_test, y_test))


1.0
0.8398058252427184


In [ ]:
df['y']

In [26]:
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import roc_auc_score

bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=200
)
bag.fit(X_train, y_train)
print(f"train score: {bag.score(X_train, y_train)}")
print(f"test score: {bag.score(X_test, y_test)}")

y_proba = bag.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC: {auc:.4f}")
# random forest "ko'p tree + tasodif*2 + birlashitirish = barqaror bashorat"

train score: 1.0
test score: 0.8919902912621359
ROC AUC: 0.7592


In [31]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features="sqrt",
    oob_score=True,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"test: {model.score(X_test, y_test)}")
print(f"oob score: {model.oob_score_}")

test: 0.9004854368932039
oob score: 0.8952959028831563


In [37]:
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_test, y_pred)
print(matrix)

[[726   8]
 [ 74  16]]


In [35]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

y_proba = model.predict_proba(X_test)[:, 1]
print(f"ROC AUC -> {roc_auc_score(y_test, y_proba):.4f}")

              precision    recall  f1-score   support

          no       0.91      0.99      0.95       734
         yes       0.67      0.18      0.28        90

    accuracy                           0.90       824
   macro avg       0.79      0.58      0.61       824
weighted avg       0.88      0.90      0.87       824

ROC AUC -> 0.7444


In [43]:
# Boosting  -  ketma-ket o'qitish
# har yangi model oldingilarining xatolarini tuzatadi: AdaBoost misolida

from sklearn.ensemble import AdaBoostClassifier

boost = AdaBoostClassifier(n_estimators=200, random_state=42)
boost.fit(X_test, y_test)
print(f"train score: {boost.score(X_train, y_train)}")
print(f"test score: {boost.score(X_test, y_test)}")

y_proba = boost.predict_proba(X_test)[:, 1]
print(f"ROC AUC score -> {roc_auc_score(y_test, y_proba):.4f}")

train score: 0.8959028831562974
test score: 0.9053398058252428
ROC AUC score -> 0.8671
